### 과일 Data를 CNN으로 Image 분류

In [22]:
import numpy as np

fruits = np.load("../Data/fruits.npy")
fruits.shape

(300, 100, 100)

In [23]:
# Target 만들기
target = np.concatenate(
    [
        np.zeros(100), # Apple
        np.ones(100), # pine apple
        np.full(100, 2) #banana
    ]
)

#### train과 test

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
train = fruits.reshape(-1, 100, 100, 1) / 255.0

train_data, test_data, train_target, test_target = \
        train_test_split(
            train,
            target,
            test_size = 0.2,
            random_state=42
        )

In [26]:
# 크기 확인
print(train_data.shape)
print(test_data.shape)
print(train_target.shape)
print(test_target.shape)

(240, 100, 100, 1)
(60, 100, 100, 1)
(240,)
(60,)


### CNN만들기

In [27]:
from tensorflow.keras import layers, models
from tensorflow import keras

In [28]:
# ----------------------------------------
# [2] CNN 모델 구조 정의
# ----------------------------------------
model = models.Sequential([
    # 첫 번째 Convolution 레이어 (특징 추출)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(100, 100, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # 두 번째 Convolution 레이어
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # 세 번째 Convolution 레이어
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Dense 레이어로 넘어가기 위해 3D 특징맵을 1D로 평탄화
    layers.Flatten(),
    
    # Fully Connected (Dense) 레이어
    layers.Dense(64, activation='relu'),
    
    # 출력 레이어 (클래스가 3개이므로 노드 수 3, 다중 분류를 위해 softmax 사용)
    layers.Dense(3, activation='softmax')
])

c:\Users\tjoeun\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 98, 98, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 21, 21, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       409,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 465,603 (1.78 MB)

 Trainable params: 465,603 (1.78 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [31]:
checkpoint_cb = keras.callbacks.ModelCheckpoint("../Data/best_cnn_fruit_model.keras")

early_stopping_cb= keras.callbacks.EarlyStopping(
                    patience=2,
                    restore_best_weights=True                  
)

history = model.fit(
            train_data,
            train_target,
            epochs=50,
            batch_size=32,
            validation_split=0.2,
            callbacks = [checkpoint_cb, early_stopping_cb]
)

Epoch 1/50


6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 138ms/step - accuracy: 0.5104 - loss: 0.9057 - val_accuracy: 0.5417 - val_loss: 0.7342
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.7917 - loss: 0.5088 - val_accuracy: 0.9167 - val_loss: 0.2266
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - accuracy: 0.8490 - loss: 0.2820 - val_accuracy: 0.8958 - val_loss: 0.2172
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.9635 - loss: 0.1310 - val_accuracy: 1.0000 - val_loss: 0.0798
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - accuracy: 0.9896 - loss: 0.0605 - val_accuracy: 1.0000 - val_loss: 0.0094
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - accuracy: 1.0000 - loss: 0.0108 - val_accuracy: 1.0000 - val_loss: 0.0139
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 1.0000 - loss: 6.9412e-04 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoc

In [35]:
pred = model.predict(
    train_data[:1]
)
pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step


array([[1.2377730e-08, 2.3903328e-09, 1.0000000e+00]], dtype=float32)

In [36]:
# 글자로 변경하기
import numpy as np
classes = ['apple', 'pine apple', 'banana']
len(classes)

3

In [37]:
classes[np.argmax(pred)]

'banana'

In [39]:
test_scaled = test_data.reshape(-1, 100, 100, 1) / 255.0
model.evaluate(test_scaled, test_target)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3667 - loss: 8.4959


[8.495927810668945, 0.36666667461395264]